IMPORTS

In [81]:
import tqdm
import random
import pathlib
import itertools
import collections

import cv2
import einops
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers


from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models


In [2]:
file_path = "/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video"

In [ ]:
import os
from pathlib import Path
from typing import List, Dict

def list_files_from_unzipped_dataset(root_dir: str) -> Dict[str, List[str]]:
    """
    List all video files grouped by class (exercise) from the unzipped dataset folder.

    Args:
        root_dir: Path to the unzipped root folder (e.g., './gym-workoutexercises-video'
                  or wherever you extracted it). This folder should contain
                  subfolders like 'pushups', 'squats', etc.

    Returns:
        Dictionary where keys are class names (exercise/folder names)
        and values are lists of full paths to the video files in that class.
    """
    root_path = Path(root_dir).resolve()
    if not root_path.is_dir():
        raise ValueError(f"Directory not found: {root_path}")

    class_files = {}

    # Walk through immediate subdirectories (each should be a class)
    for class_dir in root_path.iterdir():
        if class_dir.is_dir():
            class_name = class_dir.name  # e.g., 'pushups', 'squats'
            video_paths = []

            # Collect .mp4 files (or adjust extension if needed)
            for file_path in class_dir.glob('*.mp4'):
                video_paths.append(str(file_path.resolve()))

            # Optionally sort for consistency
            video_paths.sort()

            if video_paths:  # only add if there are videos
                class_files[class_name] = video_paths

    return class_files


# Example usage:
# dataset_root = "/path/to/your/unzipped/folder"   # ← change this
# files_by_class = list_files_from_unzipped_dataset(dataset_root)

# Then print summary:
# for cls, paths in files_by_class.items():
#     print(f"{cls}: {len(paths)} videos")

In [15]:
all_files = list_files_from_unzipped_dataset(file_path)

Your Task/Prompt: In a new cell, call your function, compute/print the stats (e.g., "Class 'deadlift' has 32 videos"), and inspect 1-2 videos' metadata. What do you notice about class balance or video variations? Update your notebook and share the output/stats if you want feedback before Step 2.

In [21]:
for key, value in all_files.items():
    print(f"Workout {key} has {len(value)} files")


Workout deadlift has 32 files
Workout hammer curl has 12 files
Workout tricep Pushdown has 50 files
Workout squat has 23 files
Workout push-up has 56 files
Workout tricep dips has 16 files
Workout lat pulldown has 51 files
Workout barbell biceps curl has 62 files
Workout chest fly machine has 28 files
Workout incline bench press has 33 files
Workout leg extension has 19 files
Workout shoulder press has 13 files
Workout t bar row has 14 files
Workout decline bench press has 6 files
Workout bench press has 61 files
Workout lateral raise has 31 files
Workout pull Up has 26 files
Workout plank has 6 files
Workout leg raises has 15 files
Workout hip thrust has 14 files
Workout romanian deadlift has 10 files
Workout russian twist has 12 files


In [ ]:
total = sum(len(value) for value in all_files.values()); print(f"Total videos: {total}")

Total videos: 590


In [23]:
sorted(all_files.items(), key=lambda x: len(x[1]), reverse=True)

[('barbell biceps curl',
  ['/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_1.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_10.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_11.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_12.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_13.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_14.mp4',
   '/Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/bar

In [24]:
class_videos = all_files

Using cv2 to understand metadata of videofiles

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Cannot open video")
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_sec = frame_count / fps if fps > 0 else 0

    cap.release()
    return {
        'frames': frame_count,
        'fps': fps,
        'width': width,
        'height': height,
        'duration_sec': duration_sec
    }

# Test on 1–2 paths
info1 = get_video_info(class_videos['bench press'][0])
print("Bench press example:", info1)

Bench press example: {'frames': 58, 'fps': 29.97002997002997, 'width': 1920, 'height': 1080, 'duration_sec': 1.9352666666666667}


In [27]:
info2 = get_video_info(class_videos['plank'][0])
print("Plank example:", info2)

Plank example: {'frames': 333, 'fps': 29.97002997002997, 'width': 1280, 'height': 720, 'duration_sec': 11.1111}


In [28]:
info3 = get_video_info(class_videos['barbell biceps curl'][0])
print("Barbell example:", info3)

Barbell example: {'frames': 114, 'fps': 29.97002997002997, 'width': 1920, 'height': 1080, 'duration_sec': 3.8038}


PREPROCESSING CONSTANTS

In [29]:
NUM_FRAMES = 16    # or 32 — try small first
IMG_SIZE = 224

In [ ]:
def preprocess_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None

    indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)

    frames = []
    for idx in indices:
        # clamp index (your repeat-last-frame logic)
        actual_idx = min(idx, total_frames - 1)

        # Move to the desired frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, actual_idx)

        # Read the frame
        ret, frame = cap.read()
        if not ret:
            print(f"Failed to read frame {actual_idx} from {video_path}")
            continue

        # Convert BGR (OpenCV default) → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        resized_frame = cv2.resize(frame_rgb, (IMG_SIZE, IMG_SIZE))
        frames.append(resized_frame)

        # For now — just collect (we'll resize next)
        # frames.append(frame_rgb)



    cap.release()

    # Convert list → numpy array
    frames_array = np.array(frames)   # shape should be (num_read, H, W, 3)

    frames_array = frames_array.astype(np.float32) / 255.0

    print(f"Read {len(frames_array)} frames from {video_path}")
    print(f"Shape: {frames_array.shape}")

    return tf.convert_to_tensor(frames_array)


In [48]:
bench_press_ex = class_videos['bench press'][2]
preprocess_video(bench_press_ex)

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/bench press/bench press_11.mp4
Shape: (16, 224, 224, 3)


<tf.Tensor: shape=(16, 224, 224, 3), dtype=float32, numpy=
array([[[[1.        , 0.53333336, 0.63529414],
         [1.        , 0.53333336, 0.63529414],
         [1.        , 0.53333336, 0.63529414],
         ...,
         [1.        , 0.4392157 , 0.5764706 ],
         [1.        , 0.43137255, 0.5803922 ],
         [1.        , 0.42352942, 0.57254905]],

        [[1.        , 0.53333336, 0.63529414],
         [1.        , 0.53333336, 0.63529414],
         [1.        , 0.53333336, 0.63529414],
         ...,
         [1.        , 0.4392157 , 0.5764706 ],
         [1.        , 0.43137255, 0.5803922 ],
         [1.        , 0.42352942, 0.57254905]],

        [[1.        , 0.53333336, 0.63529414],
         [1.        , 0.53333336, 0.63529414],
         [1.        , 0.5372549 , 0.6431373 ],
         ...,
         [1.        , 0.4392157 , 0.5764706 ],
         [1.        , 0.43137255, 0.5803922 ],
         [1.        , 0.42352942, 0.57254905]],

        ...,

        [[0.9882353 , 0.56078434,

In [49]:
plank_ex = class_videos['plank'][1]
preprocess_video(plank_ex)

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/plank/plank_3.mp4
Shape: (16, 224, 224, 3)


<tf.Tensor: shape=(16, 224, 224, 3), dtype=float32, numpy=
array([[[[0.5686275 , 0.89411765, 1.        ],
         [0.57254905, 0.8980392 , 1.        ],
         [0.5803922 , 0.90588236, 1.        ],
         ...,
         [0.00784314, 0.        , 0.        ],
         [0.        , 0.        , 0.        ],
         [0.        , 0.        , 0.        ]],

        [[0.5686275 , 0.89411765, 1.        ],
         [0.57254905, 0.8980392 , 1.        ],
         [0.5803922 , 0.90588236, 1.        ],
         ...,
         [0.00784314, 0.        , 0.        ],
         [0.        , 0.        , 0.        ],
         [0.        , 0.        , 0.        ]],

        [[0.5686275 , 0.89411765, 1.        ],
         [0.57254905, 0.8980392 , 1.        ],
         [0.5803922 , 0.90588236, 1.        ],
         ...,
         [0.00392157, 0.        , 0.        ],
         [0.        , 0.        , 0.        ],
         [0.        , 0.        , 0.        ]],

        ...,

        [[0.13333334, 0.12156863,

In [50]:
result = preprocess_video(plank_ex)
print("Shape:", result.shape)
print("Dtype:", result.dtype)
print("Min / Max:", tf.reduce_min(result).numpy(), tf.reduce_max(result).numpy())

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/plank/plank_3.mp4
Shape: (16, 224, 224, 3)
Shape: (16, 224, 224, 3)
Dtype: <dtype: 'float32'>
Min / Max: 0.0 1.0


In [51]:
result = preprocess_video(bench_press_ex)
print("Shape:", result.shape)
print("Dtype:", result.dtype)
print("Min / Max:", tf.reduce_min(result).numpy(), tf.reduce_max(result).numpy())

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/bench press/bench press_11.mp4
Shape: (16, 224, 224, 3)
Shape: (16, 224, 224, 3)
Dtype: <dtype: 'float32'>
Min / Max: 0.003921569 1.0


In [52]:

classes = sorted(class_videos.keys())           # alphabetical order → stable indices
class_to_idx = {cls: i for i, cls in enumerate(classes)}

print("Classes and their indices:")
for cls, idx in class_to_idx.items():
    print(f"{idx:2d} → {cls} ({len(class_videos[cls])} videos)")

# Now build the two big lists
all_video_paths = []
all_labels = []

for class_name, paths in class_videos.items():
    label_id = class_to_idx[class_name]
    all_video_paths.extend(paths)
    all_labels.extend([label_id] * len(paths))

print(f"\nTotal videos collected: {len(all_video_paths)}")
print(f"Total labels: {len(all_labels)}")
print(f"Example (first 3):")
for i in range(min(3, len(all_video_paths))):
    print(f"  {all_video_paths[i]} → label {all_labels[i]} ({classes[all_labels[i]]})")

Classes and their indices:
 0 → barbell biceps curl (62 videos)
 1 → bench press (61 videos)
 2 → chest fly machine (28 videos)
 3 → deadlift (32 videos)
 4 → decline bench press (6 videos)
 5 → hammer curl (12 videos)
 6 → hip thrust (14 videos)
 7 → incline bench press (33 videos)
 8 → lat pulldown (51 videos)
 9 → lateral raise (31 videos)
10 → leg extension (19 videos)
11 → leg raises (15 videos)
12 → plank (6 videos)
13 → pull Up (26 videos)
14 → push-up (56 videos)
15 → romanian deadlift (10 videos)
16 → russian twist (12 videos)
17 → shoulder press (13 videos)
18 → squat (23 videos)
19 → t bar row (14 videos)
20 → tricep Pushdown (50 videos)
21 → tricep dips (16 videos)

Total videos collected: 590
Total labels: 590
Example (first 3):
  /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_1.mp4 → label 3 (deadlift)
  /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/dead

In [ ]:
def load_video(path, label):
    def _inner_preprocess(p_tensor):
        # Force conversion to Python string safely
        path_str = p_tensor.numpy().decode('utf-8')
        # Now call your normal function
        return preprocess_video(path_str)

    video = tf.py_function(
        _inner_preprocess,
        inp=[path],
        Tout=tf.float32,
        name='preprocess_video_pyfunc'  # optional, helps debugging
    )

    video.set_shape([NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3])
    return video, label

In [65]:
paths_tensor = tf.constant(all_video_paths)
labels_tensor = tf.constant(all_labels, dtype=tf.int32)

dataset = tf.data.Dataset.from_tensor_slices((paths_tensor, labels_tensor))

In [66]:
# Apply loading + preprocessing
dataset = dataset.map(
    load_video,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Basic settings
dataset = dataset.cache()               # if you have enough RAM
dataset = dataset.shuffle(buffer_size=300)
dataset = dataset.batch(4)              # small batch for testing
dataset = dataset.prefetch(tf.data.AUTOTUNE)

# Quick test: get one batch
for videos, lbls in dataset.take(1):
    print("Batch shape:", videos.shape)           # should be (4, 16, 224, 224, 3)
    print("Labels:", lbls.numpy())
    print("Class names:", [classes[l] for l in lbls.numpy()])

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_1.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_12.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_16.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_15.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_10.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Us

2026-02-18 14:03:06.582086: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:27: Filling up shuffle buffer (this may take a while): 29 of 300


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_10.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_17.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_12.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_13.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_14.mp4
Shape: (16,

2026-02-18 14:03:17.359725: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:27: Filling up shuffle buffer (this may take a while): 68 of 300


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_31.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_32.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_34.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_33.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_35.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tr

...

2026-02-18 14:04:33.530867: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_1.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_10.mp4
Shape: (16, 224, 224, 3)
Batch shape: (4, 16, 224, 224, 3)
Labels: [20  8  3 14]
Class names: ['tricep Pushdown', 'lat pulldown', 'deadlift', 'push-up']
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_12.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_

2026-02-18 14:04:36.266756: W tensorflow/core/kernels/data/cache_dataset_ops.cc:917] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_14.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_17.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_18.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_2.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_19.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/work

2026-02-18 14:04:37.363925: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
try:
    print("Starting batch fetch...")
    for videos, labels in dataset.take(1):
        print("\n=== Batch loaded successfully ===")
        print("Videos shape:", videos.shape)
        print("Labels (integers):", labels.numpy())

        # Convert integers back to class names
        batch_class_names = [classes[int(lbl)] for lbl in labels.numpy()]
        print("Class names in this batch:", batch_class_names)

        # Optional: look at pixel stats of first video
        first_video_first_frame = videos[0][0]
        print("First frame pixel range (min/max):",
              tf.reduce_min(first_video_first_frame).numpy(),
              tf.reduce_max(first_video_first_frame).numpy())

except Exception as e:
    print("Caught error during batch loading:")
    print(e)

Starting batch fetch...
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_16.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_12.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_1.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_14.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/deadlift/deadlift_15.mp4
Shape: (16, 224, 224, 3)

2026-02-18 14:11:38.010248: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:27: Filling up shuffle buffer (this may take a while): 33 of 300


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_12.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_17.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_18.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_14.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/hammer curl/hammer curl_13.mp4
Shape: (16,

2026-02-18 14:11:57.534494: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:27: Filling up shuffle buffer (this may take a while): 108 of 300


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_27.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_25.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_26.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_28.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_7.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/squat/squat_29.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleet

...

Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_29.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_27.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_28.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_26.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_3.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_f

2026-02-18 14:13:00.005116: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_1.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_10.mp4
Shape: (16, 224, 224, 3)

=== Batch loaded successfully ===
Videos shape: (4, 16, 224, 224, 3)
Labels (integers): [14 14  0  0]
Class names in this batch: ['push-up', 'push-up', 'barbell biceps curl', 'barbell biceps curl']
First frame pixel range (min/max): 0.0 1.0
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_11.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_13.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauricee

2026-02-18 14:13:02.663342: W tensorflow/core/kernels/data/cache_dataset_ops.cc:917] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_15.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_17.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_18.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_2.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/chest fly machine/chest fly machine_19.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/work

In [ ]:
# Stratified split to preserve class distribution
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_video_paths,
    all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)

print(f"Train: {len(train_paths)} videos | Val: {len(val_paths)} videos")

Train: 472 videos | Val: 118 videos


In [71]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (tf.constant(train_paths, dtype=tf.string),
     tf.constant(train_labels, dtype=tf.int32))
).map(load_video, num_parallel_calls=tf.data.AUTOTUNE)\
 .cache()\
 .shuffle(400)\
 .batch(8)\
 .prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (tf.constant(val_paths, dtype=tf.string),
     tf.constant(val_labels, dtype=tf.int32))
).map(load_video, num_parallel_calls=tf.data.AUTOTUNE)\
 .batch(8)\
 .cache()\
 .prefetch(tf.data.AUTOTUNE)

In [ ]:
def augment(video, label):
    # video shape: (16, 224, 224, 3)

    # Random horizontal flip — same flip decision for ALL frames (consistent across time)
    def flip_fn(frames):
        # frames: (16, 224, 224, 3)
        should_flip = tf.random.uniform(()) > 0.5
        return tf.cond(
            should_flip,
            lambda: tf.image.flip_left_right(frames),   # flip whole video stack
            lambda: frames
        )

    # Apply flip across the entire time dimension at once
    video = tf.map_fn(
        lambda frame: tf.image.flip_left_right(frame) if tf.random.uniform(()) > 0.5 else frame,
        video,
        fn_output_signature=tf.float32
    )

    # Alternative: flip the whole stack with one decision (better for consistency)
    # video = flip_fn(video)

    # Other 2D augmentations (brightness, contrast, etc.) — apply per frame or to whole stack
    video = tf.image.random_brightness(video, max_delta=0.1)
    video = tf.image.random_contrast(video, lower=0.8, upper=1.2)

    # Clip to valid range if needed (especially after brightness/contrast)
    video = tf.clip_by_value(video, 0.0, 1.0)   # if you're in [0,1]
    # or clip to [-mean/std range] if normalized

    return video, label

In [78]:
train_dataset = train_dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

In [79]:
train_dataset

<_ParallelMapDataset element_spec=(TensorSpec(shape=(None, 16, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [82]:


model = models.Sequential([
    layers.Input(shape=(16, 224, 224, 3)),
    layers.Conv3D(32, (3,3,3), activation='relu'),
    layers.MaxPooling3D((1,2,2)),
    layers.Conv3D(64, (3,3,3), activation='relu'),
    layers.MaxPooling3D((1,2,2)),
    layers.GlobalAveragePooling3D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 14, 222, 222,   │         2,624 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 14, 111, 111,   │             0 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 12, 109, 109,   │        55,360 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 12, 54, 54, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │         2,838 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,142 (270.09 KB)

 Trainable params: 69,142 (270.09 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 15s/step - accuracy: 0.1031 - loss: 2.9982 Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/bench press/bench press_41.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/barbell biceps curl/barbell biceps curl_41.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_5.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/push-up/push-up_47.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/saintlouisleetokyowest-bot/ai_form/test_models/workoutfitness-video/tricep Pushdown/tricep pushdown_23.mp4
Shape: (16, 224, 224, 3)
Read 16 frames from /Users/mauriceengel/code/sai